# 01 · RAG 파이프라인 기본 구성

Phase 3 — `data/docs/` 코퍼스를 청킹·임베딩하여 **pgvector**에 적재하고,
Ollama LLM과 LangChain LCEL 체인으로 end-to-end 응답까지 확인한다.

## 진행 단계
1. **문서 로드 & 청킹** — `RecursiveCharacterTextSplitter` (chunk_size=500, overlap=50)
2. **임베딩** — `dragonkue/BGE-m3-ko`
3. **pgvector 적재 & 유사도 검색** 확인
4. **Ollama LLM 연결** — `ChatOllama` (`gemma4-e2b`)
5. **RAG 체인(LCEL)** — 검색 + 생성 end-to-end 확인

> 사전 준비: `docker compose up -d` (pgvector healthy), `ollama pull gemma4-e2b`, `.env` 작성

## 0. 환경 설정

`.env`에서 접속 정보를 읽고, 파이프라인 전역 하이퍼파라미터를 한곳에 모아 둔다.

In [1]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv("../.env")  # ANTHROPIC / Ollama / DB 접속 정보

# --- 경로 ---
DOCS_DIR = Path("../data/docs")
GOLDEN_PATH = Path("../data/golden_set.json")

# --- 청킹 ---
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

# --- 임베딩 / 컬렉션 ---
EMBED_MODEL = "dragonkue/BGE-m3-ko"
COLLECTION_NAME = "rag_docs_bge_m3_ko"

# --- 빠른 검증용: 일부 문서만 인덱싱 (None = 전체 1,417개) ---
# BGE-m3-ko를 CPU로 전체 인덱싱하면 수십 분 소요될 수 있다. 우선 작은 값으로 동작 확인 권장.
MAX_DOCS = 200

# --- pgvector 접속 문자열 (langchain-postgres v2 PGEngine → psycopg3) ---
PG_CONN = (
    f"postgresql+psycopg://{os.getenv('PGVECTOR_USER', 'user')}:"
    f"{os.getenv('PGVECTOR_PASSWORD', 'test123!')}@"
    f"{os.getenv('PGVECTOR_HOST', 'localhost')}:"
    f"{os.getenv('PGVECTOR_PORT', '5432')}/"
    f"{os.getenv('PGVECTOR_DB', 'ragdb')}"
)

# Phase 4·5에서 재사용할 골든셋 (검색/응답 예시 쿼리로도 사용)
golden_set = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))
print(f"골든셋: {len(golden_set)}개 / 문서 디렉토리: {DOCS_DIR}")
print(f"임베딩: {EMBED_MODEL} / 컬렉션: {COLLECTION_NAME}")

골든셋: 50개 / 문서 디렉토리: ../data/docs
임베딩: dragonkue/BGE-m3-ko / 컬렉션: rag_docs_bge_m3_ko


## 1. 문서 로드 & 청킹

`data/docs/`의 title별 텍스트 파일을 로드하고 `RecursiveCharacterTextSplitter`로 청킹한다.
각 청크에는 출처 추적을 위해 `title` 메타데이터를 부여한다.

In [2]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# .txt 로드는 파일을 읽어 Document로 감싸는 것뿐이라 TextLoader(langchain-community, sunset 예정)
# 없이 pathlib로 직접 처리한다. 출처 추적용으로 title(=파일명) 메타데이터를 부여한다.
doc_paths = sorted(DOCS_DIR.glob("*.txt"))
if MAX_DOCS:
    doc_paths = doc_paths[:MAX_DOCS]

documents = [
    Document(
        page_content=p.read_text(encoding="utf-8"),
        metadata={"title": p.stem, "source": str(p)},  # 파일명(=title)을 출처로 기록
    )
    for p in doc_paths
]

print(f"로드한 문서: {len(documents)}개 (MAX_DOCS={MAX_DOCS})")

로드한 문서: 200개 (MAX_DOCS=200)


In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],  # 문단 → 문장 → 단어 순으로 분할 시도
)
chunks = splitter.split_documents(documents)

lengths = [len(c.page_content) for c in chunks]
print(f"청크 수: {len(chunks)}  (문서당 평균 {len(chunks) / len(documents):.1f}개)")
print(f"청크 길이 — min {min(lengths)} / 평균 {sum(lengths) // len(lengths)} / max {max(lengths)}")
print("-" * 60)
print("예시 청크 메타:", chunks[0].metadata)
print(chunks[0].page_content[:200], "...")

청크 수: 1750  (문서당 평균 8.8개)
청크 길이 — min 15 / 평균 359 / max 500
------------------------------------------------------------
예시 청크 메타: {'title': '12국_목록', 'source': '../data/docs/12국_목록.txt'}
각 나라에 하나씩 존재하는 최고위 신수. 왕기를 의지하여 자신의 주인을 선택한다. 그 뒤 왕과 함께 나라로 돌아가 신하가 되어 곁에서 왕을 돕는다. 기린의 성격은 기본적으로 자국민의 기질을 기준으로, 민의가 구체화 한 것이라고 여겨지고 있다. 성정은 인이며 자비심이 깊다. 싸움을 싫어하며 피의 부정으로 병이 들기도 한다. 자신의 주인 이외에는 결코 머리를  ...


## 2. 임베딩 모델 준비

`dragonkue/BGE-m3-ko`를 로드한다. 코사인 유사도 검색을 위해 `normalize_embeddings=True`로 정규화하고,
사용 가능한 가속기(CUDA/MPS/CPU)를 자동 선택한다.

In [4]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

# 동작 확인 — 임베딩 차원 출력
probe = embeddings.embed_query("임베딩 차원 확인용 테스트 문장")
print(f"device={device} / 임베딩 차원: {len(probe)}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

device=mps / 임베딩 차원: 1024


## 3. pgvector 적재 & 유사도 검색

`langchain-postgres` v2의 `PGVectorStore`로 청크를 임베딩하여 pgvector에 적재한다.
구버전 `PGVector`(고정 2테이블: `langchain_pg_collection` + `langchain_pg_embedding`)와 달리,
v2는 **엔진 생성 → 전용 테이블 초기화 → 팩토리 생성**의 2단계 구성이며 `table_name`마다 테이블 1개를 만든다.

- `collection_name` → `table_name`, `embedding` → `embedding_service`
- `pre_delete_collection=True` 역할은 `init_vectorstore_table(..., overwrite_existing=True)`가 대신한다(멱등)
- `vector_size`는 임베딩 차원과 반드시 일치해야 한다 (BGE-m3-ko = 1024)

> 적재는 모든 청크를 임베딩하므로 시간이 걸린다. CPU라면 `MAX_DOCS`를 작게 두고 먼저 확인할 것.

In [ ]:
from langchain_postgres import PGEngine, PGVectorStore

# langchain-postgres v2: PGVector(단일 추상화) → PGEngine + PGVectorStore(2단계 구성)
engine = PGEngine.from_connection_string(url=PG_CONN)

# 전용 테이블을 명시적으로 생성한다. vector_size는 임베딩 차원과 반드시 일치(BGE-m3-ko=1024).
# overwrite_existing=True → 재실행 시 테이블 재생성(멱등). 구버전 pre_delete_collection 역할.
engine.init_vectorstore_table(
    table_name=COLLECTION_NAME,
    vector_size=len(probe),  # = 1024 (위 셀 embed_query 결과 차원)
    overwrite_existing=True,
)

vectorstore = PGVectorStore.create_sync(
    engine=engine,
    table_name=COLLECTION_NAME,  # 구버전 collection_name → table_name
    embedding_service=embeddings,  # 구버전 embedding → embedding_service
)
vectorstore.add_documents(chunks)
print(f"pgvector 적재 완료: {len(chunks)}개 청크 → 테이블 {COLLECTION_NAME!r}")

pgvector 적재 완료: 1750개 청크 → 테이블 'rag_docs_bge_m3_ko'


In [6]:
# 유사도 검색 동작 확인 — 골든셋 첫 질문으로 top-3 조회
query = golden_set[0]["question"]
ground_truth = golden_set[0]["ground_truth"]
hits = vectorstore.similarity_search_with_score(query, k=3)

print(f"질문: {query}")
print(f"정답(ground_truth): {ground_truth}")
print("=" * 70)
for rank, (doc, score) in enumerate(hits, 1):
    title = doc.metadata.get("title")
    preview = doc.page_content[:160].replace("\n", " ")
    print(f"[{rank}] score={score:.4f}  title={title}")
    print(preview, "...")
    print("-" * 70)

질문: 구식 군인들의 월급인 쌀에 모래와 돌멩이가 들어가있던 사건을 말미암아 일어난 사태의 이름은?
정답(ground_truth): 임오군란
[1] score=0.5901  title=경신_대기근
4월 27일에 여섯 번째로 기우제를 지냈다. 그러나 이틀 뒤인 4월 29일, 평안도에 한재(旱災), 즉 재난 급의 가뭄이 일어났다는 보고가 접수되었다. 5월 2일에 임금이 내린 하교를 보면, “들판이 모두 타버려서 밀 보리를 수확할 수 없게 되었고 파종 시기를 놓치게 되었다”고 언급하고 ...
----------------------------------------------------------------------
[2] score=0.5932  title=경신_대기근
비록 이미 농사는 망쳤지만, 5월 말에 내린 비로 가뭄이 끝나는가 싶더니, 이번에는 홍수가 찾아왔다. 6월 1일, 전라도에 큰 비가 연일 내려 들판이 강이 될 지경이라는 전라 감사의 보고가 접수되었다. 6월 8일에는 경상도에 참혹한 홍수가 졌다는 경상 감사의 보고가 올라왔다. 이어 6월 ...
----------------------------------------------------------------------
[3] score=0.6176  title=5.18_광주_민주화_운동
1980년 5월 초부터 신군부 세력의 정치 관여를 반대하기 위해, 학생과 시민 10만여 명이 모여 서울역에서 시위를 벌였고 5월 15일 시위대 대열 속에 속했던 청년 한 명이 버스를 탈취하여 저지선을 돌파, 전경에 돌진하여 전경 이성재 일경이 사망하고 4명이 중상을 입는 사고가 발생했다 ...
----------------------------------------------------------------------


## 4. Ollama LLM 연결

로컬 Ollama 서버의 `gemma4-e2b` 모델을 `ChatOllama`로 연결한다 (재현성을 위해 `temperature=0`).

In [9]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=os.getenv("OLLAMA_MODEL", "gemma4:e2b"),
    base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
    temperature=0,
)

# 연결 확인
print(llm.invoke("한 문장으로 자기소개해줘.").content)

저는 Google DeepMind에서 개발한 오픈 웨이트 대규모 언어 모델인 Gemma 4입니다.


## 5. RAG 체인 (LCEL) — end-to-end 확인

pgvector retriever + Ollama LLM을 `langchain-core` LCEL로 묶는다(stuff 방식).
문맥 밖 환각을 줄이기 위해 한국어 프롬프트로 "문맥 근거 답변"을 강제하고,
검색 근거(`source_documents`)도 함께 반환하도록 구성한다.

> 과거의 `RetrievalQA`는 LangChain 1.x에서 deprecated(레거시 `langchain-classic`)이므로 LCEL로 대체한다.

In [10]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# RetrievalQA는 deprecated(langchain-classic)라 langchain-core 기반 LCEL로 직접 구성한다.
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_template(
    "당신은 주어진 문맥(context)에만 근거해 한국어로 간결히 답하는 어시스턴트입니다.\n"
    '문맥에서 답을 찾을 수 없으면 "문맥에서 답을 찾을 수 없습니다."라고 답하세요.\n\n'
    "문맥:\n{context}\n\n"
    "질문: {question}\n"
    "답변:"
)


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


answer_chain = prompt | llm | StrOutputParser()

# 입력(질문 문자열) → 검색(source_documents) + 생성(answer)을 함께 반환.
# RetrievalQA의 return_source_documents=True와 동일하게 근거 문서를 보존한다.
qa_chain = RunnableParallel(
    question=RunnablePassthrough(),
    source_documents=retriever,
).assign(
    answer=lambda x: answer_chain.invoke(
        {"context": format_docs(x["source_documents"]), "question": x["question"]}
    )
)
print("RAG 체인 구성 완료 (LCEL, retriever k=4, stuff 방식)")

RAG 체인 구성 완료 (LCEL, retriever k=4, stuff 방식)


In [11]:
# end-to-end 실행 — 골든셋 질문에 대한 RAG 응답 + 근거 문서
sample = golden_set[0]
result = qa_chain.invoke(sample["question"])

answer = result["answer"].strip()
print(f"질문      : {sample['question']}")
print(f"정답      : {sample['ground_truth']}")
print(f"RAG 응답  : {answer}")
print("=" * 70)
print("근거 문서:")
for i, doc in enumerate(result["source_documents"], 1):
    title = doc.metadata.get("title")
    snippet = doc.page_content[:80].strip()
    print(f"  [{i}] {title} — {snippet}...")

질문      : 구식 군인들의 월급인 쌀에 모래와 돌멩이가 들어가있던 사건을 말미암아 일어난 사태의 이름은?
정답      : 임오군란
RAG 응답  : 문맥에서 답을 찾을 수 없습니다.
근거 문서:
  [1] 경신_대기근 — 4월 27일에 여섯 번째로 기우제를 지냈다. 그러나 이틀 뒤인 4월 29일, 평안도에 한재(旱災), 즉 재난 급의 가뭄이 일어났다는 보고가 접수...
  [2] 경신_대기근 — 비록 이미 농사는 망쳤지만, 5월 말에 내린 비로 가뭄이 끝나는가 싶더니, 이번에는 홍수가 찾아왔다. 6월 1일, 전라도에 큰 비가 연일 내려...
  [3] 5.18_광주_민주화_운동 — 1980년 5월 초부터 신군부 세력의 정치 관여를 반대하기 위해, 학생과 시민 10만여 명이 모여 서울역에서 시위를 벌였고 5월 15일 시위대...
  [4] 경신_대기근 — 네 번째 기우제를 지냈던 4월 9일에 양심합에서 임금과 대신들이 나눈 대화를 살펴보면, 비가 너무 오지 않아 도저히 파종을 할 수 없는 지경이라...


In [12]:
# 여러 질문에 대해 한눈에 확인 (앞 5개)
for item in golden_set[:5]:
    out = qa_chain.invoke(item["question"])
    rag = out["answer"].strip()[:120]
    print(f"Q: {item['question']}")
    print(f"  RAG: {rag}")
    print(f"  GT : {item['ground_truth']}")
    print("-" * 70)

Q: 구식 군인들의 월급인 쌀에 모래와 돌멩이가 들어가있던 사건을 말미암아 일어난 사태의 이름은?
  RAG: 문맥에서 답을 찾을 수 없습니다.
  GT : 임오군란
----------------------------------------------------------------------
Q: 백련교도의 난 때 수세에 몰린 왕총아가 전투 도중 절벽에서 뛰어내려 자진한 때는 언제인가?
  RAG: 1798년(가경 3년)
  GT : 1798년
----------------------------------------------------------------------
Q: 고려대학교 교호에 등장하는 몇 사람의 이름이 등장하는가?
  RAG: 네 사람의 이름이 등장한다.
  GT : 네 사람
----------------------------------------------------------------------
Q: 주병진이 12년 만에 출연한 프로그램은?
  RAG: 문맥에서 답을 찾을 수 없습니다.
  GT : 무릎팍도사
----------------------------------------------------------------------
Q: 서재필은 어떤 사람들에게 도움을 요청했는가?
  RAG: 문맥에서 답을 찾을 수 없습니다.
  GT : 한인대회에 참석한 [[미국인
----------------------------------------------------------------------


## 정리

- `data/docs/` → 청킹 → `dragonkue/BGE-m3-ko` 임베딩 → **pgvector** 적재까지 RAG 인덱싱 파이프라인 구성
- pgvector 유사도 검색과 Ollama `gemma4-e2b` 기반 **LCEL RAG 체인** end-to-end 응답 확인
- 다음 단계(Phase 4): 동일 체인에 골든셋을 흘려 **RAGAS** 4대 지표로 품질 평가 (`02_ragas_eval.ipynb`)

> 전체 코퍼스로 평가하려면 `MAX_DOCS = None`으로 재인덱싱한 뒤 Phase 4를 진행한다.